In [4]:
%pip install unsloth transformers datasets trl peft accelerate bitsandbytes

  Using cached unsloth-2026.8.3-py3-none-any.whl.metadata (72 kB)
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached trl-1.9.2-py3-none-any.whl.metadata (12 kB)
  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.50.0-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached unsloth_zoo-2026.8.3-py3-none-any.whl.metadata (33 kB)
  Using cached wheel-0.47.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached tyro-1.0.15-py3-none-any.whl.metadata (12 kB)
  Using cached protobuf-7.35.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached xformers-0.0.35-py39-none-win_amd64.whl.metadata (1.0 kB)
  Using cached triton_windows-3.7.1.post27-cp312-cp312-win_amd64.whl.metadata (1.8 kB)
  Using cached sentencepiece-0.2.2-cp312-cp312-


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


d:\Projects\Healthcare Scheduling System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0808 02:15:51.103000 37332 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.3: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1660. Num GPUs = 1. Max memory: 6.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:51<00:00,  5.62it/s]


In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Torch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True
GPU: NVIDIA GeForce GTX 1660


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

Unsloth 2026.8.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
from datasets import load_dataset

ds = load_dataset(
    "json",
    data_files="../data/datasets.json",
    split="train"
)

print(ds[0])

{'question': 'What is the Barangay Bagumbayan Health Center?', 'answer': 'The Barangay Bagumbayan Health Center is a government healthcare facility that provides primary healthcare services, health education, disease prevention, and medical consultations to residents of Barangay Bagumbayan, Taguig City.'}


In [5]:
def format_dataset(example):
    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}


dataset = ds.map(format_dataset)

Map: 100%|██████████| 450/450 [00:00<00:00, 1415.75 examples/s]


In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="outputs",

    num_train_epochs=2,
    learning_rate=1e-4,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,

    logging_steps=10,
    save_strategy="epoch",

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    optim="adamw_8bit",

    report_to="none",

    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,

    max_length=1024,
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Packing train dataset: 100%|██████████| 450/450 [00:00<00:00, 586.90 examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [10]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9 | Num Epochs = 2 | Total steps = 4
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("phi3-dolly-lora")
tokenizer.save_pretrained("phi3-dolly-lora")

In [ ]:
model.save_pretrained_gguf(
    "gguf_model",
    tokenizer,
    quantization_method="q4_k_m",
)